In [ ]:
import os
import geopandas as gpd
import numpy as np
import rioxarray
from pystac_client import Client
from odc.stac import load, configure_rio

# =====================================================
# CONFIGURATION 
# =====================================================

SHAPEFILE = "/run/media/vincent/Extreme Pro/Data/shapefiles/Kenya.shp"
YEAR = "2024"

RAW_DIR = "outputs/geomad_raw"
IDX_DIR = "outputs/geomad_indices"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(IDX_DIR, exist_ok=True)

STAC_URL = "https://explorer.digitalearth.africa/stac"

# =====================================================
# DEA S3 CONFIGURATION
# =====================================================

configure_rio(
    cloud_defaults=True,
    aws={"aws_unsigned": True},
    AWS_S3_ENDPOINT="s3.af-south-1.amazonaws.com",
)

# =====================================================
# LOAD AOI
# =====================================================

print("Loading shapefile...")
aoi = gpd.read_file(SHAPEFILE)


if aoi.crs is None:
    raise ValueError("Shapefile CRS is missing")



aoi = aoi.to_crs("EPSG:4326")

bbox = tuple(aoi.total_bounds)

print("AOI bounds:")
print(bbox)



# =====================================================
# SEARCH DEA STAC
# =====================================================

print("Searching GeoMAD...")

catalog = Client.open(STAC_URL)

search = catalog.search(
    collections=["gm_s2_annual"],
    bbox=bbox,
    datetime=f"{YEAR}-01-01/{YEAR}-12-31",
)

items = list(search.items())

if len(items) == 0:
    raise ValueError("No GeoMAD items found")

print(f"Found {len(items)} item(s)")

# =====================================================
# BANDS
# =====================================================

bands = [
    "B02","B03", "B04","B05","B06","B07","B08",
    "B8A","B11","B12","EMAD","SMAD","BCMAD","COUNT",
]

# =====================================================
# LOAD DATA (NO DASK)
# =====================================================

print("Loading GeoMAD dataset...")

ds = load(
    items,
    bands=bands,
    bbox=bbox,
    resolution=10,
    chunks=None
)

if "time" in ds.dims:
    ds = ds.squeeze("time", drop=True)

print(ds)

# =====================================================
# CRS
# =====================================================

if ds.odc.crs is None:
    raise ValueError("Dataset CRS missing")

ds = ds.rio.write_crs(str(ds.odc.crs))

# =====================================================
# CLIP TO SHAPEFILE
# =====================================================

print("Clipping to AOI...")

aoi_proj = aoi.to_crs(ds.odc.crs)

ds_clip = ds.rio.clip(
    aoi_proj.geometry,
    aoi_proj.crs,
    drop=True
)

print(ds_clip)

# =====================================================
# SAVE RAW BANDS
# =====================================================

print("Saving raw GeoMAD layers...")

for band in bands:

    if band not in ds_clip.data_vars:
        print(f"Missing band: {band}")
        continue

    outfile = os.path.join(
        RAW_DIR,
        f"{band}_kenya.tif"
    )

    ds_clip[band].rio.to_raster(
        outfile,
        compress="LZW",
        tiled=True
    )

    print("Saved:", outfile)

# =====================================================
# SCALE REFLECTANCE
# =====================================================

scale = 10000.0

B02 = ds_clip["B02"].astype("float32") / scale
B03 = ds_clip["B03"].astype("float32") / scale
B04 = ds_clip["B04"].astype("float32") / scale
B08 = ds_clip["B08"].astype("float32") / scale
B11 = ds_clip["B11"].astype("float32") / scale
B12 = ds_clip["B12"].astype("float32") / scale

eps = 1e-8

# =====================================================
# DERIVED INDICES
# =====================================================

indices = {}

indices["NDVI"] = (
    (B08 - B04)
    / (B08 + B04 + eps)
)

indices["EVI"] = (
    2.5 * (
        (B08 - B04)
        /
        (
            B08
            + 6 * B04
            - 7.5 * B02
            + 1
            + eps )
    )
)
indices["SAVI"] = (
    1.5 *
    (
        (B08 - B04)
        /
        (B08 + B04 + 0.5 + eps)
    )
)

indices["MSAVI"] = (
    (
        2 * B08 + 1
        -
        np.sqrt(
            np.maximum(
                0,
                (2 * B08 + 1) ** 2
                - 8 * (B08 - B04)
            )
        )
    )
    / 2
)

indices["NDMI"] = (
    (B08 - B11)
    /
    (B08 + B11 + eps)
)
indices["NDWI"] = (
    (B03 - B08)
    /
    (B03 + B08 + eps)
)

indices["NBR"] = (
    (B08 - B12)
    /
    (B08 + B12 + eps)
)

indices["NBR2"] = (
    (B11 - B12)
    /
    (B11 + B12 + eps)
)

# =====================================================
# SAVE INDICES
# =====================================================

print("Saving indices...")

for name, da in indices.items():

    da = da.where(
        (da >= -1)
        & (da <= 1)
    )

    outfile = os.path.join(
        IDX_DIR,
        f"{name}_kenya.tif"
    )

    da.rio.to_raster(
        outfile,
        dtype="float32",
        compress="LZW",
        tiled=True
    )
    print("Saved:", outfile)

# =====================================================
# FINISHED
# =====================================================

print("\n================================")
print("GeoMAD download complete")
print("================================")
print("Raw layers:", RAW_DIR)
print("Indices:", IDX_DIR)

Loading shapefile...
AOI bounds:
(np.float64(33.91018061800018), np.float64(-4.724106299999967), np.float64(41.91011899900016), np.float64(4.620000000000502))
Searching GeoMAD...
Found 126 item(s)
Loading GeoMAD dataset...


Aborting load due to failure while reading: s3://deafrica-services/gm_s2_annual/1-0-0/x221/y081/2024--P1Y/gm_s2_annual_x221y081_2024--P1Y_B02.tif:1


RasterioIOError: Read failed. See previous exception for details.

### Downloadind at 1 km resolution for Kenya

In [2]:
import os
import warnings

import geopandas as gpd
import numpy as np
import rioxarray
from pystac_client import Client
from odc.stac import load, configure_rio

warnings.filterwarnings("ignore")

# =====================================================
# CONFIGURATION
# =====================================================

SHAPEFILE = "/run/media/vincent/Extreme Pro/Data/shapefiles/Kenya.shp"
YEAR = "2024"

RAW_DIR = "outputs/geomad_raw_kenya"
IDX_DIR = "outputs/geomad_indices_kenya"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(IDX_DIR, exist_ok=True)

STAC_URL = "https://explorer.digitalearth.africa/stac"

# =====================================================
# GDAL / AWS SETTINGS
# =====================================================

os.environ["AWS_NO_SIGN_REQUEST"] = "YES"

os.environ["GDAL_HTTP_MAX_RETRY"] = "10"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "5"

os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"

configure_rio(
    cloud_defaults=True,
    aws={"aws_unsigned": True},
    AWS_S3_ENDPOINT="s3.af-south-1.amazonaws.com",
)

# =====================================================
# LOAD AOI
# =====================================================

print("Loading shapefile...")

aoi = gpd.read_file(SHAPEFILE)

if aoi.crs is None:
    raise ValueError("Shapefile CRS is missing")

aoi = aoi.to_crs("EPSG:4326")

bbox = tuple(aoi.total_bounds)

print("AOI bounds:")
print(bbox)

# =====================================================
# SEARCH DEA STAC
# =====================================================

print("\nSearching GeoMAD...")

catalog = Client.open(STAC_URL)

search = catalog.search(
    collections=["gm_s2_annual"],
    bbox=bbox,
    datetime=f"{YEAR}-01-01/{YEAR}-12-31",
)

items = list(search.items())

if len(items) == 0:
    raise ValueError("No GeoMAD items found")

print(f"Found {len(items)} item(s)")

# =====================================================
# BANDS NEEDED FOR INDICES
# =====================================================

bands = [
    "B02",  # Blue
    "B03",  # Green
    "B04",  # Red
    "B08",  # NIR
    "B11",  # SWIR1
    "B12",  # SWIR2
]

# =====================================================
# LOAD GEOMAD
# =====================================================

print("\nLoading GeoMAD at 1 km resolution...")

try:

    ds = load(
        items,
        bands=bands,
        bbox=bbox,
        resolution=250,      # 1 km
        fail_on_error=False,
        chunks={"x": 512, "y": 512},
    )

except Exception as e:
    raise RuntimeError(f"Failed loading GeoMAD: {e}")

if "time" in ds.dims:
    ds = ds.squeeze("time", drop=True)

print(ds)

# =====================================================
# CRS
# =====================================================

if ds.odc.crs is None:
    raise ValueError("Dataset CRS missing")

ds = ds.rio.write_crs(str(ds.odc.crs))

# =====================================================
# CLIP
# =====================================================

print("\nClipping to Kenya boundary...")

aoi_proj = aoi.to_crs(ds.odc.crs)

ds = ds.rio.clip(
    aoi_proj.geometry,
    aoi_proj.crs,
    drop=True,
)

print(ds)

# =====================================================
# SAVE RAW BANDS
# =====================================================

print("\nSaving raw bands...")

for band in bands:

    outfile = os.path.join(
        RAW_DIR,
        f"{band}_Kenya_100_m.tif"
    )

    ds[band].rio.to_raster(
        outfile,
        dtype="float32",
        compress="LZW",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    print(f"Saved: {outfile}")

# =====================================================
# SCALE REFLECTANCE
# =====================================================

scale = 10000.0
eps = 1e-8

B02 = ds["B02"].astype("float32") / scale
B03 = ds["B03"].astype("float32") / scale
B04 = ds["B04"].astype("float32") / scale
B08 = ds["B08"].astype("float32") / scale
B11 = ds["B11"].astype("float32") / scale
B12 = ds["B12"].astype("float32") / scale

# =====================================================
# INDICES
# =====================================================

print("\nCalculating indices...")

indices = {}

# NDVI
indices["NDVI"] = (
    (B08 - B04)
    /
    (B08 + B04 + eps)
)

# EVI
indices["EVI"] = (
    2.5
    *
    (
        (B08 - B04)
        /
        (
            B08
            + 6.0 * B04
            - 7.5 * B02
            + 1.0
            + eps
        )
    )
)

# SAVI
indices["SAVI"] = (
    1.5
    *
    (
        (B08 - B04)
        /
        (B08 + B04 + 0.5 + eps)
    )
)

# MSAVI
indices["MSAVI"] = (
    (
        2 * B08
        + 1
        -
        np.sqrt(
            np.maximum(
                0,
                (2 * B08 + 1) ** 2
                - 8 * (B08 - B04)
            )
        )
    )
    / 2
)

# NDMI
indices["NDMI"] = (
    (B08 - B11)
    /
    (B08 + B11 + eps)
)

# NDWI
indices["NDWI"] = (
    (B03 - B08)
    /
    (B03 + B08 + eps)
)

# NBR
indices["NBR"] = (
    (B08 - B12)
    /
    (B08 + B12 + eps)
)

# NBR2
indices["NBR2"] = (
    (B11 - B12)
    /
    (B11 + B12 + eps)
)

# =====================================================
# SAVE INDICES
# =====================================================

print("\nSaving indices...")

for name, da in indices.items():

    da = da.where(
        np.isfinite(da)
    )

    da = da.where(
        (da >= -1.0)
        &
        (da <= 1.0)
    )

    outfile = os.path.join(
        IDX_DIR,
        f"{name}_Kenya_100m.tif"
    )

    da.rio.to_raster(
        outfile,
        dtype="float32",
        compress="LZW",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    print(f"Saved: {outfile}")

# =====================================================
# COMPLETE
# =====================================================

print("\n========================================")
print("GeoMAD processing complete")
print("========================================")
print(f"Raw bands folder : {RAW_DIR}")
print(f"Indices folder   : {IDX_DIR}")
print("Resolution       : 100 m")
print("========================================")

Loading shapefile...
AOI bounds:
(np.float64(33.91018061800018), np.float64(-4.724106299999967), np.float64(41.91011899900016), np.float64(4.620000000000502))

Searching GeoMAD...
Found 126 item(s)

Loading GeoMAD at 1 km resolution...
<xarray.Dataset> Size: 177MB
Dimensions:      (y: 4765, x: 3088)
Coordinates:
  * y            (y) float64 38kB 5.889e+05 5.886e+05 ... -6.019e+05 -6.021e+05
  * x            (x) float64 25kB 3.272e+06 3.272e+06 ... 4.043e+06 4.044e+06
    spatial_ref  int32 4B 6933
Data variables:
    B02          (y, x) uint16 29MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    B03          (y, x) uint16 29MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    B04          (y, x) uint16 29MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    B08          (y, x) uint16 29MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    B11          (y, x) uint16 29MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    B12          (y, x) uint16 29MB dask.array<chun